# 🍌 Train YOLOv11 - Deteksi Kematangan Buah Sawo

Notebook ini akan melatih model **YOLOv11** untuk mendeteksi 2 kelas kematangan buah sawo:
- **Matang** (class 0)
- **Mentah / Belum Matang** (class 1)

### Yang perlu di-upload:
1. `data.zip` - Dataset gambar & label dalam format YOLO

### Hasil yang didapat:
- `best.pt` & `last.pt` - Model weights
- `best.onnx` - Model untuk deploy web
- Grafik training, confusion matrix, dll

> **PENTING:**
> - Pakai **yolo11n** (nano) supaya ringan di browser (~5MB ONNX)
> - Export ONNX pakai `imgsz=320` (optimal untuk real-time webcam di browser)

## 1. Cek GPU

In [ ]:
!nvidia-smi

## 2. Upload Dataset

Upload file **data.zip**

In [ ]:
from google.colab import files

print('Upload file data.zip ...')
uploaded = files.upload()

## 3. Install Requirements

In [ ]:
!pip install ultralytics

In [ ]:
# Unzip images to a custom data folder
!unzip -q /content/data.zip -d /content/custom_data

!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

## 4. Configure Training

In [ ]:
import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found at {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = [line.strip() for line in f.readlines() if line.strip()]

  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': len(classes),
      'names': classes
  }

  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

create_data_yaml('/content/custom_data/classes.txt', '/content/data.yaml')

print('\nFile contents:\n')
!cat /content/data.yaml

## 5. Train Model

Menggunakan **yolo11n** (nano) supaya model ringan di browser.

In [ ]:
# Training tetap pakai imgsz=640 (resolusi tinggi untuk belajar fitur)
# Nanti export ONNX pakai 320 untuk kecepatan di browser
!yolo detect train data=/content/data.yaml model=yolo11n.pt epochs=60 imgsz=640

## 6. Test Model

In [ ]:
import glob

# Auto-detect folder hasil training terakhir
train_dirs = sorted(glob.glob('/content/runs/detect/train*'))
run_dir = train_dirs[-1] if train_dirs else '/content/runs/detect/train'
best_pt = f'{run_dir}/weights/best.pt'

print(f'Menggunakan hasil training dari: {run_dir}')
print(f'Model: {best_pt}')

!yolo detect predict model={best_pt} source=/content/data/validation/images save=True

In [ ]:
import glob
from IPython.display import Image, display

pred_dirs = sorted(glob.glob('/content/runs/detect/predict*'))
pred_dir = pred_dirs[-1] if pred_dirs else '/content/runs/detect/predict'

for image_path in glob.glob(f'{pred_dir}/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')

## 7. Lihat Hasil Training

In [ ]:
from IPython.display import Image, display
import os

print(f'Hasil training di: {run_dir}')

plots = [
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'results.png',
    'labels.jpg',
    'train_batch0.jpg',
    'val_batch0_pred.jpg',
]

for fname in plots:
    fpath = os.path.join(run_dir, fname)
    if os.path.exists(fpath):
        print(f'\n{fname}')
        display(Image(filename=fpath, width=800))
    else:
        print(f'{fname} - tidak ditemukan')

## 8. Export ke ONNX

> **PENTING:** `imgsz=320` = optimal untuk real-time webcam di browser
> - 320 → ~40-80ms, 15-25 FPS
> - 512 → ~150-200ms, 5-7 FPS
> - 640 → ~300-500ms, 2-3 FPS

In [ ]:
# imgsz=320 untuk performa optimal di browser
# HARUS SAMA dengan MODEL_SIZE di src/lib/yolo/session.ts
!yolo export model={best_pt} format=onnx imgsz=320 opset=14 simplify=True

## 9. Download Semua Hasil Training

In [ ]:
import shutil, os

if os.path.exists('/content/my_model'):
    shutil.rmtree('/content/my_model')
os.makedirs('/content/my_model', exist_ok=True)

# Copy weights
!cp {run_dir}/weights/best.pt /content/my_model/best.pt
!cp {run_dir}/weights/last.pt /content/my_model/last.pt

# Copy ONNX
!cp {run_dir}/weights/best.onnx /content/my_model/best.onnx

# Copy semua grafik, CSV, dan file hasil training
!cp -r {run_dir} /content/my_model/train_results

# Lihat isi folder
print('Isi folder my_model:')
!find /content/my_model -type f | sort

# Zip dan download
%cd /content/my_model
!zip /content/my_model.zip best.pt last.pt best.onnx
!zip -r /content/my_model.zip train_results
%cd /content

print('\nDownloading my_model.zip ...')

from google.colab import files
files.download('/content/my_model.zip')

---

## Selesai!

### File yang didapat di `my_model.zip`:
| File | Keterangan |
|------|------------|
| `best.pt` | Model terbaik (PyTorch) |
| `last.pt` | Model epoch terakhir |
| `best.onnx` | Model untuk deploy web (**yolo11n, imgsz=320**) |
| `train_results/` | Grafik, confusion matrix, results.csv, dll |

### Performa di browser:
| imgsz | Inference | FPS | Akurasi |
|-------|-----------|-----|---------|
| 320 | ~40-80ms | **15-25** | Cukup baik |
| 512 | ~150-200ms | 5-7 | Baik |
| 640 | ~300-500ms | 2-3 | Terbaik |

### Deploy ke SawoVision:
1. Extract `my_model.zip`
2. Copy `best.onnx` ke `public/models/best.onnx`
3. Reload aplikasi
4. Test di halaman `/detect`